A notebook to fine-tune a GLiNER model based on [this example](https://github.com/urchade/GLiNER/blob/main/examples/finetune.ipynb).

## Env Install

```shell
conda create -n gliner_finetune python=3.10     
conda activate gliner_finetune
conda install gliner accelerate seqeval datasets
```

## Data and Libraries

In [5]:
import json
import random

from datasets import load_dataset

from seqeval.metrics.sequence_labeling import get_entities

import re

from collections import Counter
# import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Dict

from dataset_processing import *

ModuleNotFoundError: No module named 'matplotlib'

Loading is based on [er_combine_data](CWED4ETA/GLINER/er_combine_data.ipynb) notebook.

### Specific Dataset Loading

#### Climate-Change-NER (IBM)

In [3]:
ibmccner = load_ibmccner(["train", "validation"])

Loading 'ibm-research/Climate-Change-NER' with splits: ['train', 'validation']


KeyboardInterrupt: 

#### BiodivNER

In [2]:
biodivner = load_biodivner(["train", "validation"])

Loading '/home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER' with splits: ['train', 'validation']
Converting from BIO tags to char spans.
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/dev.csv...
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/train.csv...
Converting from char spans to token spans.


#### CWED4ETA

In [103]:
cwed4eta = load_cwed4eta()

Loading '/home/p0l3/RAD/CLIRENER/CliReNER/DATA/LABEL_STUDIO/project-30-at-2025-11-14-12-19-2a7464a5.json' with no splits.
COnverting to desirable char spans.
COnverting from char spans to token spans.


In [2]:
cwed4eta_hf = load_dataset("P0L3/CliReNER_v_1_1_28_SILVER")
labels = cwed4eta_hf["train"].features["ner_tags"][0].names
# cwed4eta_hf["train"].features["ner_tags"].feature.names # Depending on some library version

### Data Preprocessing

In [ ]:



train_dataset = hf_dataset_to_gliner_format(cwed4eta_hf["train"], labels)
test_dataset = hf_dataset_to_gliner_format(cwed4eta_hf["validation"], labels)

## Model Fine-Tuning

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"

import torch
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

c:\Users\Desktop\miniforge3\envs\clirener_finetune\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model = GLiNER.from_pretrained("gliner-community/gliner_medium-v2.5")

Fetching 10 files: 100%|██████████| 10/10 [00:02<00:00,  3.45it/s]


In [7]:
# use it for better performance, it mimics original implementation but it's less memory efficient
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)

In [8]:
# Optional: compile model for faster training
model.to(device)
print("done")

done


In [9]:
# calculate number of epochs
num_steps = 4000
batch_size = 8
data_size = len(train_dataset)
num_batches = data_size // batch_size
num_epochs = max(1, num_steps // num_batches)

training_args = TrainingArguments(
    output_dir="models/GLiNER_med_v2_5/CliReNER_v_1_1_28_SILVER",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear", #cosine
    warmup_ratio=0.1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=num_epochs,
    eval_strategy="steps", # PC1, # evaluation_strategy="steps",
    save_steps = 100,
    save_total_limit=10,
    dataloader_num_workers = 0,
    use_cpu = False,
    report_to="none",
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

trainer.train()

C:\Users\Desktop\AppData\Local\Temp\ipykernel_15616\2355970038.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## Inference

In [3]:
trained_model = GLiNER.from_pretrained("models/GLiNER_med_v2_1/CLIRENER_V1/checkpoint-4012", load_tokenizer=True)

config.json not found in C:\Users\ANDRIJA_RAD\CLIRENER\CliReNER\FINETUNES\GLINER\models\GLiNER_med_v2_1\CLIRENER_V1\checkpoint-4012


In [8]:
from dataset_processing import CLIRENER_LABELS_V1

text = """Resulting regional commitments would be around 10% higher than the global signal for the vulnerable Pacific region, mainly due to higher relative Antarctic contributions."""
LABELS = CLIRENER_LABELS_V1
# Labels for entity prediction
labels = list(LABELS)
# ["Person", "Award"] # for v2.1 use capital case for better performance

# Perform entity prediction
entities = trained_model.predict_entities(text, labels, threshold=0.1)

# Display predicted entities and their labels
print(text)
for entity in entities:
    print(entity["text"], "=>", entity["label"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Resulting regional commitments would be around 10% higher than the global signal for the vulnerable Pacific region, mainly due to higher relative Antarctic contributions.
Resulting => Other
regional commitments => Policy
10% => Quantity
higher => Quantity
global signal => Physical Phenomenon
vulnerable => Other
Pacific region => Location
higher => Quantity
relative => Quantity
Antarctic contributions => Other


## Evaluation

In [3]:
from dataset_processing import CLIRENER_LABELS_V1, hf_dataset_to_gliner_format
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"

import torch
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset
from datasets import load_dataset


In [6]:
trained_model = GLiNER.from_pretrained("models/GLiNER_med_v2_1/CLIRENER_V1/checkpoint-4012", load_tokenizer=True)

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

trained_model = trained_model.to(device)
dataset = load_dataset("P0L3/CliReNER_v_1_1_28_SILVER")

labels = dataset["train"].features["ner_tags"][0].names
test_ds = hf_dataset_to_gliner_format(dataset["test"], labels)


config.json not found in C:\Users\ANDRIJA_RAD\CLIRENER\CliReNER\FINETUNES\GLINER\models\GLiNER_med_v2_1\CLIRENER_V1\checkpoint-4012


AttributeError: 'list' object has no attribute 'feature'

In [14]:
test_ds

[{'tokenized_text': ['However',
   ',',
   'powering',
   'EVs',
   'requires',
   'cathode',
   'materials',
   'with',
   'a',
   'higher',
   'energy',
   'density',
   '(',
   '>',
   '200',
   'mAhg',
   '−',
   '1',
   ')',
   '.'],
  'ner': [[3, 3, 'Physical Artefact'],
   [5, 6, 'Chemical'],
   [10, 11, 'Quantity'],
   [13, 13, 'Quantity'],
   [14, 17, 'Quantity']],
  'id': '1'},
 {'tokenized_text': ['For',
   'this',
   'animal',
   ',',
   'a',
   'serum',
   'phenobarbital',
   'target',
   'range',
   'of',
   '20',
   '–',
   '30',
   'μg',
   '/',
   'mL',
   'was',
   'achievable',
   'with',
   'a',
   'dose',
   'of',
   '1',
   '.',
   '5',
   'mg',
   '/',
   'kg',
   'once',
   'daily',
   'followed',
   'by',
   'therapeutic',
   'monitoring',
   ',',
   'and',
   'this',
   'is',
   'a',
   'reasonable',
   'recommended',
   'concentration',
   'and',
   'initial',
   'dose',
   'for',
   'clinicians',
   'treating',
   'this',
   'species',
   '.'],
  'ner': [[1,

In [16]:
evaluation_results = trained_model.evaluate(
    test_ds, flat_ner=True, entity_types=list(CLIRENER_LABELS_V1)
)

In [17]:
evaluation_results

('P: 76.01%\tR: 83.38%\tF1: 79.52%\n', np.float64(0.7952314165497896))